# Construct-Validity Demo — Maritime Intent Probe

Reproduces the two headline figures from the
[Maritime-Intent-Probe](https://github.com/Cacapice/Maritime-Intent-Probe) construct-validity result
on a free Colab T4 in roughly 15–20 minutes.

**What this shows.** A preregistered search for a linear "adversarial intent" direction in
Pythia-1.4B residual streams was blocked by its own bias controls: **BC1** established that in these
minimal-pair stimuli, intent and surface form are perfectly confounded — the single token swap that
defines an adversarial pair moves both at once, and downstream policy is never specified. Under this
design the intent variable is not identifiable by *any* estimator.

- **Figure 1** — a held-out probe AUC (~0.76) that clears a label-permutation null decisively and,
  by construction, cannot be interpreted as intent detection.
- **Figure 2** — the per-variant diagnostic: content-safe normalization (NFKC fold, confusable fold,
  zero-width strip, delimiter collapse) collapses 4/8 encoding families to chance random-direction
  separability, localizing their "signal" to Unicode surface; the non-foldable families remain
  confounded per BC1.

> **Interpretation policy (inherited from the OSF amendment).** No result in this notebook is
> evidence about adversarial intent. All findings characterize the representational geometry induced
> by a construct-invalid stimulus design. The preregistered hypotheses H1–H5 remain locked and
> unevaluated on OSF.

Preregistration & amendments: [osf.io/pnaxk](https://osf.io/pnaxk/overview) ·
License: AGPL-3.0

*Runtime: Colab GPU (T4). Runtime → Change runtime type → T4 GPU before running.*

In [ ]:
# ── Cell 1: Setup (~2 min) ────────────────────────────────────────────────────
# Clone at the pinned release tag so results refer to a frozen code state.
import os, subprocess, torch

assert torch.cuda.is_available(), "Enable a GPU runtime (T4) before running."

TAG = "v1.1.0"
if not os.path.exists("/content/Maritime-Intent-Probe"):
    !git clone --branch {TAG} --depth 1 https://github.com/Cacapice/Maritime-Intent-Probe.git /content/Maritime-Intent-Probe
%cd /content/Maritime-Intent-Probe

# Colab ships recent torch and scikit-learn; the repo's exact pins (sklearn==1.5.0)
# conflict with Colab's preinstalled umap-learn/hdbscan (which require >=1.6).
# The demo only uses stable sklearn APIs, so we keep Colab's version and add
# only what's missing:
!pip install -q "transformer_lens>=3.4.0"
# For the exact-pin environment used in the full pipeline (may emit resolver
# warnings about unrelated Colab packages):  !pip install -q -r requirements.txt

# Local vault (no Drive mount needed) + model selection, via the env vars
# config.py/stage_vault.py read as their single source of truth.
os.environ["VAULT_BASE_DIR"]   = "/content/vault"
os.environ["VAULT_MODEL_NAME"] = "EleutherAI/pythia-1.4b"
os.environ["VAULT_DTYPE"]      = "float16"   # T4 is pre-Ampere: no native bf16.
# float16 also halves stage_vault.py's CPU-RAM peak — the staging subprocess
# loads the model on CPU, and the standard ~12.7 GB Colab runtime OOM-kills it
# (shown as '^C' in the output) at the bf16/fp32 peak.

from config import VaultConfig
cfg = VaultConfig()
print(f"model={cfg.model_name}  seed={cfg.seed}  vault={cfg.base_dir}")

In [ ]:
# ── Cell 2: Stage + load the model (~4 min) ──────────────────────────────────
# TransformerLens's weight processing UPCASTS TO FP32 ON CPU, so CPU staging
# peaks at ~12–15 GB for 1.4B regardless of dtype — the standard Colab runtime
# OOM-kills it (a bare '^C' in the output). Staging on the GPU routes that peak
# into the T4's 15 GB VRAM instead. Weights move back to CPU before saving, so
# the vault format is byte-identical in layout to stage_vault.py's output and
# load_model() verifies it the same way.
import json, shutil, torch
from stage_vault import sha256              # same streaming SHA-256 as vault.py

if not cfg.manifest_path.exists():
    shutil.rmtree(cfg.vault_dir, ignore_errors=True)   # clear any partial stage
    cfg.vault_dir.mkdir(parents=True, exist_ok=True)

    from transformer_lens import HookedTransformer
    staged = HookedTransformer.from_pretrained(
        cfg.model_name, device="cuda", dtype=torch.float16,
    )
    torch.save({k: v.detach().cpu() for k, v in staged.state_dict().items()},
               cfg.weights_path)
    cfg.config_path.write_text(
        json.dumps(staged.cfg.to_dict(), indent=2, default=str))
    cfg.manifest_path.write_text(json.dumps({          # written LAST = complete
        "weights": sha256(cfg.weights_path),
        "config":  sha256(cfg.config_path),
    }, indent=2))
    del staged; torch.cuda.empty_cache()               # free VRAM for load_model
    print(f"Vault staged on GPU → {cfg.vault_dir}")
else:
    print(f"Vault already staged at {cfg.vault_dir} — skipping.")

from vault import load_model
model = load_model(cfg, device="cuda")

## The confound, with your own eyes

Each minimal pair differs by a single slot substitution. That one swap changes the wording **and**
the label simultaneously — and no downstream routing decision is ever read out. The cells
(intent=0, surface=adv) and (intent=1, surface=plain) are never observed, so a probe trained on
these pairs cannot be shown to read intent rather than wording. This is BC1.

In [ ]:
# ── Cell 3: Stimuli + structural audit (~1 min) ──────────────────────────────
from environment import (
    MaritimeEnvironment, MINIMAL_PAIR_VAULT, validate_minimal_pairs_structural,
)

env = MaritimeEnvironment(model.tokenizer, seed=cfg.seed)
validate_minimal_pairs_structural()   # raises MinimalPairError on any malformed pair

payloads = env.generate_minimal_pairs()
print(f"{len(payloads)} payloads generated\n")

# Show one legit/adversarial pair side by side so the single-token confound is visible.
legit = next(p for p in payloads if p.label == 0)
adv   = next(p for p in payloads if p.label == 1)
print("LEGIT (label=0):\n", legit, "\n")
print("ADVERSARIAL (label=1):\n", adv)

In [ ]:
# ── Cell 4: Activation collection (~5 min) ───────────────────────────────────
# Reduced layer subset spanning depth to keep T4 runtime down; the full sweep
# lives in the repo's experiment pipeline.
from collector import ActivationCollector

SCAN_LAYERS = [2, 6, 10, 14, 18, 22]          # Pythia-1.4B has 24 blocks
hook_names  = [f"blocks.{i}.hook_resid_post" for i in SCAN_LAYERS]

collector = ActivationCollector(model, hook_names, batch_size=16)
acts, labels = collector.collect(payloads, env)   # masked mean-pooled, per layer
print({k: tuple(v.shape) for k, v in acts.items()})

In [ ]:
# ── Cell 5: FIGURE 1 — the seductive false positive (~4 min) ─────────────────
# Reproduces the README's headline number with the repo's OWN analysis:
# supervised_projection_with_null on the adv_full vs adv_surface regions —
# held-out LDA AUC per layer against a matched in-distribution label-permutation
# null (same CV procedure, shuffled labels). This is the correct null here:
# it measures the separation achievable from this sample/dimension ratio with
# meaningless labels, at the SAME contrast the statistic is computed on.
import numpy as np, matplotlib.pyplot as plt
from surface_geometry import supervised_projection_with_null

regions = env.generate_surface_regions(n_per_region=100, cls="fragmentation")
reg_acts = {}
for tag in ("adv_full", "adv_surface"):
    a, lab = collector.collect(regions[tag], env)
    reg_acts[tag] = a                      # dict: hook_name -> [n, d_model]

prof = {}
for h in hook_names:
    prof[h] = supervised_projection_with_null(
        {"adv_full": reg_acts["adv_full"][h], "adv_surface": reg_acts["adv_surface"][h]},
        pair=("adv_full", "adv_surface"), n_perm=200, seed=cfg.seed, layer=h,
    )

xs      = SCAN_LAYERS
ys      = [prof[h]["held_out_auc"] for h in hook_names]
null_mu = [prof[h]["null_auc_mean"] for h in hook_names]
null_sd = [prof[h]["null_auc_std"]  for h in hook_names]
null_p95= [prof[h]["null_auc_p95"]  for h in hook_names]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, ys, "o-", label="Held-out LDA AUC (adv_full vs adv_surface)")
ax.plot(xs, null_p95, "--", color="gray", label="Permutation null, 95th pct")
lo = [m - s for m, s in zip(null_mu, null_sd)]
hi = [m + s for m, s in zip(null_mu, null_sd)]
ax.fill_between(xs, lo, hi, alpha=0.15, color="gray", label="permutation null ±1 sd")
ax.axhline(0.5, color="gray", lw=0.8)
ax.set(xlabel="Residual layer", ylabel="AUC", ylim=(0.4, 1.0),
       title="Clear held-out separation — and uninterpretable as intent\n"
             "(BC1: surface and content never varied independently)")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

print("Held-out AUC per layer:", {l: round(a, 3) for l, a in zip(xs, ys)})
print("Above null P95:", {l: prof[h]["auc_above_null_p95"]
                          for l, h in zip(xs, hook_names)})

In [ ]:
# ── Cell 6: FIGURE 2 — per-variant NFKC rescue (~5 min) ──────────────────────
# Runs the per-variant null gate twice (raw / normalized) with a seeded
# random-direction probe, reproducing the geometry-level collapse the amendment
# diagnostic measures: the four exact-fold variants (homoglyph, fullwidth,
# zero-width, delimiter) collapse to chance; leetspeak/hex/rot13 are non-foldable
# by design and remain confounded; abbreviation folds only partially.
from null_validator import compare_encoding_variant_nulls_normalized

class RandomDirectionProbe:
    """Minimal seeded random-direction probe exposing the direction_for(layer)
    interface null_validator expects (mirrors the private notebook's cell 3b-1)."""
    def __init__(self, d_model: int, hook_names: list[str], seed: int = 42):
        g = torch.Generator().manual_seed(seed)
        self._dirs = {}
        for h in hook_names:
            v = torch.randn(d_model, generator=g)
            self._dirs[h] = v / v.norm()
    def direction_for(self, layer: str) -> torch.Tensor:
        return self._dirs[layer]

rand_probe = RandomDirectionProbe(model.cfg.d_model, hook_names, seed=cfg.seed)

report = compare_encoding_variant_nulls_normalized(
    model, cfg, env, collector, probe=rand_probe,
    saes=None, device="cuda", n_pairs=50, hook_names=hook_names,
)

# Bar chart: |AUC − 0.5| per encoding variant, raw vs normalized.
variants = list(report["per_variant"].keys())
raw_dev  = [report["per_variant"][v]["raw_deviation"]        for v in variants]
norm_dev = [report["per_variant"][v]["normalized_deviation"] for v in variants]
fold_set = set(report["exact_fold_variants"])

x = np.arange(len(variants)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, raw_dev,  w, label="raw null")
ax.bar(x + w/2, norm_dev, w, label="NFKC-normalized null")
ax.set_xticks(x)
ax.set_xticklabels([v + (" *" if v in fold_set else "") for v in variants],
                   rotation=30, ha="right")
ax.axhline(0.0, color="gray", lw=0.8)
ax.set(ylabel="|AUC − 0.50| (random direction)",
       title="Separability without intent: 4/8 families collapse under normalization\n"
             "(* = exact-fold variant; non-foldable families remain confounded per BC1)")
ax.legend(); plt.tight_layout(); plt.show()

print("exact_fold_all_collapsed:", report.get("exact_fold_all_collapsed"))

## The identifiability condition

A representation cannot be interpreted as encoding a semantic variable unless the design
independently varies **(a)** surface form and **(b)** the downstream consequence that
operationalizes the variable. These stimuli put all probability mass on the diagonal — every
adversarial pair changed both — so no estimator can distinguish "encodes intent" from "encodes
wording." The proposed fix (Orthogonal Intent / Action-Space Probing: fully cross plain/adv surface
with safe/unsafe graph state and read out action logits) is specified in the
[repo README](https://github.com/Cacapice/Maritime-Intent-Probe) and
[FELLOWSHIP_README](https://github.com/Cacapice/Maritime-Intent-Probe/blob/main/FELLOWSHIP_README.md).

The BC1–BC11 Construct-Validity Gate is reusable — issues and PRs welcome:
[github.com/Cacapice/Maritime-Intent-Probe](https://github.com/Cacapice/Maritime-Intent-Probe) ·
[osf.io/pnaxk](https://osf.io/pnaxk/overview)